# 08 - 修正性 RAG (CRAG)

**复杂度:** ⭐⭐⭐⭐

**应用场景:** 高风险领域(法律、医疗)、域外查询、事实核查

**核心特性:** 评估文档相关性,质量低时触发网络搜索。

**流程:**
```
查询 → 检索 → 评估相关性 → 
  如果良好: 使用检索到的文档
  如果较差: 网络搜索 + 组合来源
```

In [1]:
import sys
sys.path.append('../..')

from shared.config import create_chat_model, create_embeddings, DEFAULT_MODEL, DEFAULT_TEMPERATURE, DEFAULT_VISION_MODEL
from shared.config import HF_VECTOR_STORE_PATH, DEFAULT_MODEL
from shared.utils import load_vector_store, print_section_header, format_docs
from shared.prompts import RELEVANCE_GRADER_PROMPT, CRAG_PROMPT
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

print_section_header("设置: CRAG")

embeddings = create_embeddings()
vectorstore = load_vector_store(HF_VECTOR_STORE_PATH, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = create_chat_model(model=DEFAULT_MODEL, temperature=0)

print("✅ 设置完成!")


设置: CRAG



警告: 您正在向 HF Hub 发送未经身份验证的请求。请设置 HF_TOKEN 以获得更高的速率限制和更快的下载速度。


加载权重:   0%|          | 0/103 [00:00<?, ?it/s]

已从 d:\ZKS_Data\AI\langchain-rag-tutorial\notebooks\advanced_architectures\..\..\data\vector_stores\huggingface_embeddings 加载向量存储

langchain-openai 注入了自定义 httpx 传输以应用 `http_socket_options`,这会禁用 httpx 的代理自动检测(检测到系统代理配置)。设置 `LANGCHAIN_OPENAI_TCP_KEEPALIVE=0` 或传递 `http_socket_options=()` 以恢复默认代理行为,或提供 `openai_proxy` / 您自己的 `http_client` / `http_async_client` 以获得完全控制。


✅ 设置完成!


## 2. 相关性评估器

In [2]:
print_section_header("相关性评估器")

relevance_grader = RELEVANCE_GRADER_PROMPT | llm | StrOutputParser()

# 测试评估器
query = "什么是 RAG?"
relevant_doc = "RAG 代表检索增强生成,是一种技术..."
irrelevant_doc = "Python 是一种用于...的编程语言"

print(f"查询: '{query}'\n")
print("测试相关性评估器:")
print(f"  相关文档: {relevance_grader.invoke({'question': query, 'document': relevant_doc}).strip()}")
print(f"  不相关文档: {relevance_grader.invoke({'question': query, 'document': irrelevant_doc}).strip()}")

print("\n✓ 相关性评估器工作正常!")


相关性评估器

查询: '什么是 RAG?'

测试相关性评估器:
  相关文档: yes
  不相关文档: no

✓ 相关性评估器工作正常!


## 3. 网络搜索工具

In [3]:
print_section_header("网络搜索工具")

try:
    from langchain_community.tools.tavily_search import TavilySearchResults
    
    web_search = TavilySearchResults(max_results=3)
    print("✓ Tavily 搜索工具已初始化")
    
    # 测试
    test_results = web_search.invoke("LangChain RAG 教程")
    print(f"\n测试搜索结果: {test_results[:200]}...")
    
except ImportError as e:
    print(f"⚠️  Tavily 搜索不可用: {e}")
    print("   安装命令: pip install tavily-python")
    web_search = None
except Exception as e:
    print(f"⚠️  初始化 Tavily 时出错: {e}")
    print(f"   错误类型: {type(e).__name__}")
    web_search = None


网络搜索工具

⚠️  初始化 Tavily 时出错: 1 validation error for TavilySearchAPIWrapper
  Value error, Did not find tavily_api_key, please add an environment variable `TAVILY_API_KEY` which contains it, or pass `tavily_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
   错误类型: ValidationError


## 4. CRAG 流水线

In [4]:
print_section_header("CRAG 流水线")

def crag_retrieve(query: str, threshold: float = 0.5):
    """具有相关性评估和网络回退的 CRAG 检索。"""
    # 检索
    docs = retriever.invoke(query)
    
    # 评估相关性
    relevant_docs = []
    for doc in docs:
        grade = relevance_grader.invoke({
            "question": query,
            "document": doc.page_content[:1000]
        })
        if "yes" in grade.lower():
            relevant_docs.append(doc)
    
    relevance_ratio = len(relevant_docs) / len(docs) if docs else 0
    used_web = False
    web_results = ""
    
    # 如果相关性差则进行网络搜索
    if relevance_ratio < threshold and web_search:
        print(f"⚠️  相关性低 ({relevance_ratio:.0%}),使用网络搜索...")
        web_results = web_search.invoke(query)
        used_web = True
    
    context = format_docs(relevant_docs)
    if web_results:
        context += f"\n\n[网络搜索]\n{web_results}"
    
    return {
        "context": context,
        "input": query,
        "used_web": used_web,
        "relevance_ratio": relevance_ratio
    }

print("✓ CRAG 流水线已配置")


CRAG 流水线

✓ CRAG 流水线已配置


## 5. CRAG 链路与测试

In [5]:
print_section_header("CRAG 测试")

crag_chain = (
    RunnableLambda(crag_retrieve)
    | CRAG_PROMPT
    | llm
    | StrOutputParser()
)

# 测试 1: 域内(不应触发网络)
query1 = "RAG 中的向量存储是什么?"
print(f"\n测试 1 (域内): '{query1}'")
print("=" * 80)
response1 = crag_chain.invoke(query1)
print(response1[:250])

# 测试 2: 域外(应触发网络)
if web_search:
    query2 = "最新的 Python 版本是什么?"
    print(f"\n\n测试 2 (域外): '{query2}'")
    print("=" * 80)
    response2 = crag_chain.invoke(query2)
    print(response2[:250])

print("\n\n✅ CRAG 根据文档质量自适应!")


CRAG 测试


测试 1 (域内): 'RAG 中的向量存储是什么?'
根据上下文,**向量存储**在 RAG 中是一个围绕向量数据库的包装器,用于**存储和查询嵌入**。该过程涉及嵌入文档分割的内容并将这些嵌入插入到向量中


✅ CRAG 根据文档质量自适应!


## 总结

**优势:**
✅ 通过质量检查实现高准确性  
✅ 缺失信息时的网络回退  
✅ 对域外查询的鲁棒性  
✅ 透明度高(显示何时使用网络)  

**局限性:**
- 速度慢(评估 + 潜在的网络搜索)
- 成本高(多次 LLM 调用)
- 依赖网络搜索质量

**适用场景:**
- 高准确性要求
- 预期出现域外查询
- 法律、医疗、金融领域

**下一步:** [09_self_rag.ipynb](09_self_rag.ipynb) - 自反思 RAG